In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
            border-radius: 12px; padding: 40px 40px 30px 40px; margin: 10px 0 30px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <div style="color:#a0aec0; font-size:0.8em; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;">
    LLM Lab — Section 01b
  </div>
  <h1 style="color:white; font-size:2.2em; margin:0 0 10px 0; font-weight:700; line-height:1.2; border:none;">
    Model Datasheet<br>
    <span style="color:#63b3ed;">What We're Actually Running</span>
  </h1>
  <p style="color:#cbd5e0; font-size:1em; margin:16px 0 24px 0; max-width:600px; line-height:1.6;">
    Read the datasheet before you power on the part. Qwen3.5 isn't a pure
    transformer — it's a hybrid attention architecture. Let's crack it open.
  </p>
  <div style="display:flex; gap:16px; flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">📋 Datasheet</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">🧠 Architecture</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">📊 KV Cache Math</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">💾 Memory Budget</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">⏱ ~15 min</span>
  </div>
</div>
"""))

In [ ]:
# Quick import check — fail fast if wrong kernel
try:
    import openai, psutil
except ImportError as e:
    raise ImportError(
        f"{e}. Select the 'Python 3 (Homebrew)' kernel in Jupyter: "
        "Kernel → Change Kernel → Python 3 (Homebrew)"
    ) from None

import sys, re, json, urllib.request
import html as html_mod, markdown
from pathlib import Path
from IPython.display import HTML, display

sys.path.insert(0, str(Path('../../scripts').resolve()))
import notebook_helpers

# Discover servers (queries actual model IDs from each port)
MODELS, clients = notebook_helpers.discover_servers()

if not MODELS:
    raise RuntimeError("No MLX servers found! Start at least one on ports 8800-8809.")

# Warm up with live-updating status table
notebook_helpers.warmup_models(MODELS, clients)
notebook_helpers.init(MODELS, clients)

# Map labels to HuggingFace base model repos (for architecture config)
HF_IDS = {
    "122B-A10B": "Qwen/Qwen3.5-122B-A10B", "122B": "Qwen/Qwen3.5-122B-A10B",
    "35B-A3B": "Qwen/Qwen3.5-35B-A3B", "35B": "Qwen/Qwen3.5-35B-A3B",
    "9B": "Qwen/Qwen3.5-9B", "8B": "Qwen/Qwen3-8B", "4B": "Qwen/Qwen3.5-4B",
    "2B": "Qwen/Qwen3.5-2B", "1.7B": "Qwen/Qwen3-1.7B",
    "0.8B": "Qwen/Qwen3.5-0.8B", "0.6B": "Qwen/Qwen3-0.6B",
}
for m in MODELS:
    m["hf_id"] = HF_IDS.get(m["label"], f"Qwen/Qwen3.5-{m['label']}")

In [ ]:
# Import shared helper functions
from notebook_helpers import strip_think, _md, _render_cards, compare_models, show_metrics_table
print("Helpers loaded.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📋 Step 1: Architecture Overview</h2>
</div>

Every module in this lab uses Qwen3.5 models. Before we go deeper into tokenization, prompting, RAG, or fine-tuning, we need to understand the hardware we're building on. In embedded terms, this is the **datasheet review** before you start writing firmware.

The critical thing: **Qwen3.5 is not a pure transformer.** It's a hybrid attention architecture that mixes two fundamentally different attention mechanisms.

In [ ]:
# Fetch config.json from HuggingFace + model metadata
configs = {}
model_specs = {}  # Dynamic: total_params, active_params, disk_gb, arch

def _fetch_json(url, timeout=10):
    """Fetch JSON from URL, return dict or None."""
    try:
        return json.loads(urllib.request.urlopen(url, timeout=timeout).read())
    except Exception:
        return None

for m in MODELS:
    label = m["label"]
    
    # 1. Fetch base model config (architecture info)
    url = f"https://huggingface.co/{m['hf_id']}/resolve/main/config.json"
    print(f"Fetching {label} config from {m['hf_id']}...", flush=True)
    try:
        raw = json.loads(urllib.request.urlopen(url, timeout=10).read())
        cfg = raw.get("text_config", raw)  # params may be nested
        configs[label] = cfg
        print(f"  ✓ {cfg.get('num_hidden_layers', '?')} layers")
    except Exception as e:
        print(f"  ✗ Failed: {e}")
        configs[label] = {}
    
    # 2. Fetch BASE model safetensors for real param count
    #    (quantized repos have packed tensor counts, not logical params)
    base_api = _fetch_json(f"https://huggingface.co/api/models/{m['hf_id']}")
    base_st = (base_api or {}).get("safetensors", {})
    total_params = base_st.get("total", 0)
    
    # 3. Fetch QUANTIZED model safetensors for disk size
    quant_api = _fetch_json(f"https://huggingface.co/api/models/{m['model']}")
    quant_st = (quant_api or {}).get("safetensors", {})
    quant_params = quant_st.get("parameters", {})
    dtype_bytes = {"U8": 1, "BF16": 2, "F16": 2, "F32": 4, "U32": 4, "I32": 4}
    disk_bytes = sum(count * dtype_bytes.get(dt, 1) for dt, count in quant_params.items())
    disk_gb = disk_bytes / 1e9 if disk_bytes > 0 else 0
    
    # 4. Determine architecture and active params
    # active_params_b() handles MoE labels ("35B-A3B" → 3.0) and dense ("9B" → 9.0)
    cfg = configs[label]
    num_experts = cfg.get("num_experts", cfg.get("num_local_experts", 0)) or 0
    experts_per_tok = cfg.get("num_experts_per_tok", cfg.get("num_selected_experts", 0)) or 0
    is_moe = "-A" in label or num_experts > 1
    active_params = notebook_helpers.active_params_b(label) * 1e9
    arch = "MoE" if is_moe else "Dense"
    
    model_specs[label] = {
        "total_params": total_params,
        "active_params": active_params,
        "disk_gb": round(disk_gb, 1),
        "arch": arch,
        "num_experts": num_experts,
        "experts_per_tok": experts_per_tok,
    }
    
    total_b = total_params / 1e9
    active_b = active_params / 1e9
    if is_moe:
        print(f"  ✓ {total_b:.1f}B total, {active_b:.1f}B active ({arch}), ~{disk_gb:.1f} GB on disk")
    else:
        print(f"  ✓ {total_b:.1f}B params (Dense), ~{disk_gb:.1f} GB on disk")

# Build architecture cards
cards = ""
for m in MODELS:
    cfg = configs[m["label"]]
    spec = model_specs.get(m["label"], {})
    n_layers = cfg.get("num_hidden_layers", "?")
    n_kv_heads = cfg.get("num_key_value_heads", "?")
    n_attn_heads = cfg.get("num_attention_heads", "?")
    head_dim = cfg.get("head_dim", "?")
    hidden_size = cfg.get("hidden_size", "?")
    model_type = cfg.get("model_type", "?")
    
    num_experts = spec.get("num_experts", 0)
    experts_per_tok = spec.get("experts_per_tok", 0)
    
    delta_cfg = cfg.get("linear_attention_config", {})
    delta_heads = delta_cfg.get("num_heads", "N/A")
    delta_kv_heads = delta_cfg.get("num_kv_heads", "N/A")
    
    if isinstance(n_layers, int):
        n_gqa = n_layers // 4
        n_delta = n_layers - n_gqa
        layer_split = f"{n_delta} DeltaNet + {n_gqa} GQA"
    else:
        layer_split = "?"
    
    moe_line = ""
    if num_experts > 1:
        moe_line = f"<tr><td style='padding:4px 10px; color:#6b7280;'>MoE Experts</td><td style='padding:4px 10px; font-weight:bold;'>{num_experts} total, {experts_per_tok} active/tok</td></tr>"
    
    cards += f"""
    <div style="flex:1; min-width:280px; background:#f9fafb; border:1px solid #d1d5db;
                border-left:4px solid {m['color']}; border-radius:0 8px 8px 0; padding:14px 18px;">
      <div style="color:{m['color']}; font-weight:bold; font-size:1em; margin-bottom:10px;">
        {m['label']} — {model_type}
      </div>
      <table style="font-size:0.85em; border-collapse:collapse; width:100%;">
        <tr><td style="padding:4px 10px; color:#6b7280;">Layers</td><td style="padding:4px 10px; font-weight:bold;">{n_layers} ({layer_split})</td></tr>
        <tr><td style="padding:4px 10px; color:#6b7280;">GQA Heads</td><td style="padding:4px 10px; font-weight:bold;">{n_attn_heads} attn / {n_kv_heads} KV</td></tr>
        <tr><td style="padding:4px 10px; color:#6b7280;">DeltaNet Heads</td><td style="padding:4px 10px; font-weight:bold;">{delta_heads} heads / {delta_kv_heads} KV</td></tr>
        <tr><td style="padding:4px 10px; color:#6b7280;">Head Dim</td><td style="padding:4px 10px; font-weight:bold;">{head_dim}</td></tr>
        <tr><td style="padding:4px 10px; color:#6b7280;">Hidden Size</td><td style="padding:4px 10px; font-weight:bold;">{hidden_size}</td></tr>
        {moe_line}
      </table>
    </div>"""

display(HTML(f'<div style="display:flex; gap:16px; flex-wrap:wrap; margin:10px 0;">{cards}</div>'))
n_models = len([c for c in configs.values() if c])
print(f"\nAll {n_models} models share the same 3:1 DeltaNet-to-GQA ratio.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🔀 Step 2: The Hybrid Attention Pattern</h2>
</div>

Qwen3.5 alternates two fundamentally different attention mechanisms in a **3:1 repeating block**:

```
N × (3 × (Gated DeltaNet → FFN) → 1 × (GQA Attention → FFN))
```

- **Gated DeltaNet** (75% of layers): Linear attention with a recurrent state. Like an IIR filter — it maintains a running accumulator that gets updated with each token. O(n) per token, fixed memory.
- **GQA Attention** (25% of layers): Standard grouped-query attention with full softmax. Like RAM — random access to any previous token. O(n²) per token, KV cache grows with context.

The DeltaNet layers handle the bulk of the work cheaply. The GQA "anchor" layers every 4th position provide precise long-range lookups that linear attention can't do well.

In [ ]:
# Visualize the hybrid attention pattern for each model

for m in MODELS:
    cfg = configs[m["label"]]
    n_layers = cfg.get("num_hidden_layers", 0)
    if not isinstance(n_layers, int) or n_layers == 0:
        continue
    
    # Build layer type sequence: every 4th layer is GQA (index 3, 7, 11, ...)
    blocks = ""
    for i in range(n_layers):
        is_gqa = (i % 4 == 3)
        bg = "#2563eb" if is_gqa else "#a855f7"  # blue=GQA, purple=DeltaNet
        label = "G" if is_gqa else "D"
        # Only show labels for first 32 layers to keep it readable
        layer_type = "GQA" if is_gqa else "DeltaNet"
        if n_layers <= 64:
            blocks += f'<span style="display:inline-block; width:12px; height:20px; background:{bg}; margin:1px; border-radius:2px; font-size:6px; color:white; text-align:center; line-height:20px;" title="Layer {i}: {layer_type}">{label}</span>'
        else:
            blocks += f'<span style="display:inline-block; width:6px; height:20px; background:{bg}; margin:0.5px; border-radius:1px;" title="Layer {i}: {layer_type}"></span>'
    
    n_gqa = n_layers // 4
    n_delta = n_layers - n_gqa
    
    display(HTML(f"""
    <div style="margin:12px 0; padding:12px 16px; background:#f9fafb; border:1px solid #d1d5db; border-left:4px solid {m['color']}; border-radius:0 8px 8px 0;">
      <div style="color:{m['color']}; font-weight:bold; margin-bottom:8px;">{m['label']} \u2014 {n_layers} layers</div>
      <div style="line-height:24px; font-family:monospace;">{blocks}</div>
      <div style="margin-top:8px; font-size:0.8em; color:#6b7280;">
        <span style="color:#a855f7;">\u25a0</span> DeltaNet ({n_delta}) &nbsp;&nbsp;
        <span style="color:#2563eb;">\u25a0</span> GQA ({n_gqa}) &nbsp;&nbsp;
        Pattern repeats every 4 layers: D-D-D-G
      </div>
    </div>
    """))

print("\nThe embedded analogy: DeltaNet is like an IIR filter \u2014 a running accumulator")
print("(constant memory, fast, lossy). GQA is like RAM \u2014 full random access")
print("(growing memory, slower, lossless). The architecture uses the cheap")
print("resource for 75% of layers and the expensive resource at sync points.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📊 Step 3: KV Cache Math</h2>
</div>

The KV cache formula from Section 01 needs a correction for hybrid models. **Only the GQA layers need KV cache** — DeltaNet layers use a fixed-size recurrent state.

```
KV cache (bytes) = 2 × n_gqa_layers × n_kv_heads × head_dim × seq_len × dtype_bytes
```

The hybrid design saves 75% of KV cache memory compared to a hypothetical all-GQA model.

In [ ]:
# KV Cache calculator — hybrid vs naive (all-layers) formula

CONTEXT_LENGTHS = [8_192, 32_768, 131_072]  # 8K, 32K, 128K tokens
DTYPE_BYTES = 2  # float16

rows = ""
for m in MODELS:
    cfg = configs[m["label"]]
    n_layers = cfg.get("num_hidden_layers", 0)
    n_kv_heads = cfg.get("num_key_value_heads", 0)
    head_dim = cfg.get("head_dim", 0)
    
    if not all([n_layers, n_kv_heads, head_dim]):
        continue
    
    n_gqa = n_layers // 4  # Only GQA layers need KV cache
    
    for ctx in CONTEXT_LENGTHS:
        # Hybrid formula (actual): only GQA layers
        kv_hybrid = 2 * n_gqa * n_kv_heads * head_dim * ctx * DTYPE_BYTES
        # Naive formula: if ALL layers used GQA
        kv_naive = 2 * n_layers * n_kv_heads * head_dim * ctx * DTYPE_BYTES
        savings = (1 - kv_hybrid / kv_naive) * 100 if kv_naive > 0 else 0
        
        ctx_label = f"{ctx // 1024}K"
        rows += (
            f"<tr>"
            f"<td style='padding:6px 10px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{m['label']}</td>"
            f"<td style='padding:6px 10px; border-bottom:1px solid #e5e7eb;'>{ctx_label}</td>"
            f"<td style='padding:6px 10px; border-bottom:1px solid #e5e7eb;'>{n_gqa} GQA / {n_layers} total</td>"
            f"<td style='padding:6px 10px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{kv_hybrid / 1e9:.2f} GB</td>"
            f"<td style='padding:6px 10px; color:#dc2626; border-bottom:1px solid #e5e7eb;'>{kv_naive / 1e9:.2f} GB</td>"
            f"<td style='padding:6px 10px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{savings:.0f}%</td>"
            f"</tr>"
        )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:600px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 10px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 10px; color:white; text-align:left;">Context</th>
    <th style="padding:8px 10px; color:white; text-align:left;">Layers</th>
    <th style="padding:8px 10px; color:white; text-align:left;">Hybrid KV</th>
    <th style="padding:8px 10px; color:white; text-align:left;">Naive KV</th>
    <th style="padding:8px 10px; color:white; text-align:left;">Saved</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  Formula: 2 \u00d7 n_gqa_layers \u00d7 n_kv_heads \u00d7 head_dim \u00d7 seq_len \u00d7 2 bytes (fp16).
  \"Naive\" = if all layers used GQA attention.
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💾 Step 4: Memory Budget Calculator</h2>
</div>

Given 128 GB unified RAM with 3 models loaded: how much headroom do we have? What's the maximum safe context length before we start swapping?

In [ ]:
# Memory budget: model weights + KV cache + OS overhead

ram = psutil.virtual_memory()
total_gb = ram.total / 1e9
used_gb = (ram.total - ram.available) / 1e9
available_gb = ram.available / 1e9

# Disk sizes from dynamically fetched model_specs
disk_sizes = {m["label"]: model_specs.get(m["label"], {}).get("disk_gb", 0) for m in MODELS}
total_model_gb = sum(disk_sizes.values())

# OS + system overhead (estimate from current usage minus model weights)
os_overhead_gb = max(used_gb - total_model_gb, 4.0)  # at least 4 GB for OS

# Headroom for KV cache
kv_headroom_gb = total_gb - total_model_gb - os_overhead_gb

sizes_str = ", ".join(f"{label}={gb}" for label, gb in disk_sizes.items())
print(f"System RAM:     {total_gb:.0f} GB")
print(f"Model weights:  {total_model_gb:.1f} GB ({sizes_str})")
print(f"OS overhead:    ~{os_overhead_gb:.1f} GB")
print(f"KV headroom:    ~{kv_headroom_gb:.1f} GB")
print()

# Calculate max safe context length for each model given the headroom
rows = ""
for m in MODELS:
    cfg = configs[m["label"]]
    n_layers = cfg.get("num_hidden_layers", 0)
    n_kv_heads = cfg.get("num_key_value_heads", 0)
    head_dim = cfg.get("head_dim", 0)
    
    if not all([n_layers, n_kv_heads, head_dim]):
        continue
    
    n_gqa = n_layers // 4
    # Bytes per token of KV cache for this model
    bytes_per_token = 2 * n_gqa * n_kv_heads * head_dim * 2  # 2 for K+V, 2 for fp16
    
    # Max tokens that fit in the KV headroom
    max_tokens = int(kv_headroom_gb * 1e9 / bytes_per_token) if bytes_per_token > 0 else 0
    max_tokens_k = max_tokens / 1024
    
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{m['label']}</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{bytes_per_token:,} bytes/tok</td>"
        f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{max_tokens_k:,.0f}K tokens</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">KV Cost/Token</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Max Context (safe)</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  "Safe" = fits in remaining RAM ({kv_headroom_gb:.1f} GB) without triggering swap.
  In practice, leave extra margin — swap kills throughput.
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧩 Step 5: MoE — Why 122B Fits</h2>
</div>

Mixture-of-Experts (MoE) is why a 122B parameter model can run on a 128 GB machine. Each MoE layer contains multiple small "expert" sub-networks, but a learned router only activates a subset per token.

**The embedded analogy:** MoE is like a crossbar switch — one bus selected per cycle. All the expert weights live in memory (122B total), but only ~10B are read per forward pass. The memory footprint is the full model, but the compute (and memory bandwidth) per token is proportional to the active parameters.

In [ ]:
# MoE: total params vs active params per token
# All data comes from model_specs (fetched dynamically in Step 1)

BW_GBS = 546  # M4 Max memory bandwidth (GB/s)
BYTES_PER_PARAM = 0.5  # 4-bit quantization

rows = ""
for m in MODELS:
    spec = model_specs.get(m["label"], {})
    total = spec.get("total_params", 0)
    active = spec.get("active_params", 0)
    disk_gb = spec.get("disk_gb", 0)
    arch = spec.get("arch", "?")
    
    if total == 0:
        continue
    
    active_gb = active * BYTES_PER_PARAM / 1e9
    theoretical_tps = BW_GBS / active_gb if active_gb > 0 else 0
    
    if arch == "MoE":
        desc = f"MoE: {total/1e9:.0f}B total, {active/1e9:.0f}B active"
    else:
        desc = f"Dense: all {total/1e9:.0f}B active"
    
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{m['label']}</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{disk_gb} GB</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>{active_gb:.1f} GB</td>"
        f"<td style='padding:6px 12px; color:#16a34a; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{theoretical_tps:.0f} tok/s</td>"
        f"<td style='padding:6px 12px; color:#6b7280; font-size:0.85em; border-bottom:1px solid #e5e7eb;'>{desc}</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:600px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">On Disk</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Active/Token</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Theoretical tok/s</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Architecture</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">
  Theoretical = M4 Max bandwidth ({BW_GBS} GB/s) ÷ active weight bytes per token (4-bit).
  Real-world throughput is lower due to attention compute, KV cache reads, and scheduling overhead.
</div>
"""))

# Key insight based on what's running
moe_models = [m for m in MODELS if model_specs.get(m["label"], {}).get("arch") == "MoE"]
if moe_models:
    m = moe_models[-1]
    spec = model_specs[m["label"]]
    active_gb = spec["active_params"] * BYTES_PER_PARAM / 1e9
    print(f"\nKey insight: {m['label']} is {spec['disk_gb']} GB on disk but only reads ~{active_gb:.0f} GB of active weights per token.")
    print("That's why it achieves usable throughput despite the large model size.")
else:
    print("\nAll models are dense — every parameter is read on every token.")
    print("Throughput scales inversely with model size.")

In [ ]:
# Ask the models to explain their own architecture
results = compare_models(
    "You are a Qwen3.5 model running on Apple Silicon via MLX. In 2-3 sentences, explain your own architecture "
    "(DeltaNet vs GQA layers, MoE routing if applicable) to an embedded engineer. Use hardware analogies.",
    max_tokens=512,
)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📝 Takeaways</h2>
</div>

- **Qwen3.5 is a hybrid** — not a pure transformer. 75% DeltaNet (linear, fixed memory) + 25% GQA (full attention, growing KV cache)
- **DeltaNet is like an IIR filter** — a running accumulator. Fast, cheap, constant memory. Good for broad patterns, lossy for precise retrieval
- **GQA anchor layers are like RAM** — full random access to all previous tokens. Expensive but precise. Placed every 4th layer as sync points
- **KV cache only grows for GQA layers** — 75% savings vs a hypothetical all-attention model of the same depth
- **MoE is a crossbar switch** — 122B params on disk, ~10B active per token. The memory footprint is the full model, but bandwidth per token is proportional to active params
- **Architecture determines everything** — memory, speed, quality, max context length. Read the datasheet before you build on it

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 What's Next</h2>
</div>

- **Section 01c:** Inference Optimization — speculative decoding, prefix caching, quantization format deep dive, continuous batching
- **Section 02:** Structured output — making LLMs return JSON you can actually parse